In [1]:
import numpy as np
import plotly.graph_objects as go
%load_ext autoreload
%autoreload 2


In [2]:
#Artificially created Jet (Toy Data)
#order [pT, eta,phi]
jet_particles = np.array([
    [50.0,1.2,0.5],
    [10.0,1.3,0.6],
    [5.0, 1.1, 0.4],
    [2.0,1.5,0.8],
    [1.5,1.0,0.2]
])

#converting to the Cartesian coordinates
pt = jet_particles[:,0]
eta = jet_particles[:,1]
phi = jet_particles[:,2]

px = pt*np.cos(phi)
py = pt*np.sin(phi)
pz = pt*np.sinh(eta)

#Plotting
fig = go.Figure(data=[go.Scatter3d(
    x = px,
    y = py,
    z = pz,
    mode = 'markers',
    marker= dict(
        size = pt*0.5,
        color = pt,
        colorscale = 'Viridis',
        opacity = 0.8,
        showscale = True,
        colorbar = dict(title = 'pT (Energy)')
    )
)])



In [3]:
%pip install --upgrade nbformat ipykernel ipywidgets

Note: you may need to restart the kernel to use updated packages.


In [4]:
import sys
import subprocess

fig.update_layout(
    title = '3D Particle Jet Visualization',
    scene= dict(
        xaxis_title = 'Px (Momentum X)',
        yaxis_title = 'Py (Momentum Y)',
        zaxis_title = 'Pz (Momentum Z)'
    )
)

fig.show()

In [5]:
import h5py
import os

file_path = os.path.join(r'D:\particle-gnn\data', 'g.hdf5')

# Opening HDF5 
try:
    with h5py.File(file_path, 'r') as f:
        print("Keys in the dataset:", list(f.keys()))
        
        # extracting the first 10 jets from the dataset
        raw_jets = f['particle_features'][:10]
        
    print(f"\nSuccess! Loaded {raw_jets.shape[0]} jets.")
    print(f"Shape of one jet: {raw_jets[0].shape}") 
    
except OSError as e:
    print(f"Error: {e}\n check if the file exists and is accessible.")

Keys in the dataset: ['jet_features', 'particle_features']

Success! Loaded 10 jets.
Shape of one jet: (30, 4)


In [6]:
import torch
from torch_geometric.data import Data

def custom_knn_graph(coords, k, loop=True):
    """
    Computes the k-nearest neighbors for a set of coordinates using pure PyTorch.
    Bypasses the need for external C++ libraries like pyg-lib or torch_cluster.
    """
    # 1. Calculate pairwise distance matrix between all particles
    dist_matrix = torch.cdist(coords, coords)
    
    # 2. Find the top-k nearest neighbors
    # We use negative distances because topk finds the largest values by default
    _, col = torch.topk(-dist_matrix, k=k, dim=1)
    
    # 3. Create the edge index format required by PyTorch Geometric
    # row represents the source nodes, col represents the target nodes
    row = torch.arange(coords.size(0)).view(-1, 1).repeat(1, k)
    
    edge_index = torch.stack([row.flatten(), col.flatten()], dim=0)
    return edge_index

def create_pyg_graph(jet_features, label, k_neighbors=8):
    """
    Converts a numpy array of jet features into a PyTorch Geometric Data object.
    Assumes jet_features shape is (N, 4) with [eta, phi, pt, mask].
    """
    # 1. Filter out zero-padded particles using the mask (column index 3)
    mask = jet_features[:, 3] > 0.5 
    valid_particles = jet_features[mask]

    if len(valid_particles) == 0:
        return None 

    # 2. Extract node features (eta, phi, pt)
    node_features = valid_particles[:, 0:3]
    x = torch.tensor(node_features, dtype=torch.float)

    # 3. Create edges using the custom KNN function based on eta and phi
    coords = x[:, 0:2] 
    num_nodes = coords.size(0)
    actual_k = min(k_neighbors, num_nodes) 
    
    edge_index = custom_knn_graph(coords, k=actual_k, loop=True)

    # 4. Set the target label (e.g., 0 for Gluon, 1 for Quark)
    y = torch.tensor([label], dtype=torch.long)

    # 5. Return the standardized PyG Data object
    return Data(x=x, edge_index=edge_index, y=y)

# Test the function with the first loaded jet
first_jet_data = raw_jets[0]
jet_graph = create_pyg_graph(first_jet_data, label=0)

print("--- PyTorch Geometric Graph Object ---")
print(jet_graph)
print(f"Number of nodes (particles): {jet_graph.num_nodes}")
print(f"Number of edges: {jet_graph.num_edges}")

c:\Users\dinet\AppData\Local\Programs\Python\Python314\Lib\site-packages\torch\jit\_script.py:1485: FutureWarning: `torch.jit.script` is not supported in Python 3.14+ and may break. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


--- PyTorch Geometric Graph Object ---
Data(x=[30, 3], edge_index=[2, 240], y=[1])
Number of nodes (particles): 30
Number of edges: 240


In [7]:
from torch_geometric.data import Dataset
from torch_geometric.loader import DataLoader
import h5py
import os

class JetDataset(Dataset):
    def __init__(self, filepath, label):
        super().__init__(None,None,None)
        self.filepath = filepath
        self.label = label

        with h5py.File(self.filepath, 'r') as f:
            self.num_jets = len(f['particle_features'])

    def len(self):
        return self.num_jets
    def get(self,idx):
        with h5py.File(self.filepath,'r') as f:
            jet_data = f['particle_features'][idx]
        graph = create_pyg_graph(jet_data, label=self.label)
        if graph is None:
            return self.get((idx + 1) % self.num_jets)
        return graph
file_path = os.path.join(r'D:\particle-gnn\data', 'g.hdf5')
gluon_dataset = JetDataset(file_path, label=0)
print(f"Total jets in dataset: {len(gluon_dataset)}")

#Dataloader for batching
batch_size = 32
shuffle = True
loader = DataLoader(gluon_dataset, batch_size=batch_size, shuffle=shuffle)
for batch in loader:
    print("\n--- First Batch Details---")
    print(batch)
    print(f"Batch size (number of graphs): {batch.num_graphs}")
    print(f"Batch node features shape: {batch.x.shape}")
    break


Total jets in dataset: 177252

--- First Batch Details---
DataBatch(x=[960, 3], edge_index=[2, 7680], y=[32], batch=[960], ptr=[33])
Batch size (number of graphs): 32
Batch node features shape: torch.Size([960, 3])


In [8]:
import sys
import os 
sys.path.append(os.path.abspath(os.path.join(os.getcwd(),'..')))
sys.path.append('.')

from src.model import ParticleNetEdgeBlock 

edge_block = ParticleNetEdgeBlock(in_features=3, out_features=64)
print("ParticleNetEdgeBlock initialized successfully.")
output = edge_block(batch.x, batch.edge_index)
print(f"Input shape to ParticleNetEdgeBlock: {batch.x.shape}")
print(f"Output shape from ParticleNetEdgeBlock: {output.shape}")

ParticleNetEdgeBlock initialized successfully.
Input shape to ParticleNetEdgeBlock: torch.Size([960, 3])
Output shape from ParticleNetEdgeBlock: torch.Size([960, 64])


In [9]:

from src.model import ParticleNet

# Building the full ParticleNet model
full_model = ParticleNet(num_node_features=3, num_classes=2)

# Inserting the batch data into the model
predictions = full_model(batch.x, batch.edge_index, batch.batch)

# Output results
print(f"Shape of final predictions: {predictions.shape}")
print("\nFirst 5 Jets' Model Predictions (Logits):")
print(predictions[:5])

Shape of final predictions: torch.Size([32, 2])

First 5 Jets' Model Predictions (Logits):
tensor([[ 0.1005,  0.0773],
        [-0.0073, -0.0896],
        [ 0.0942,  0.0117],
        [ 0.1265, -0.1709],
        [-0.0350, -0.2788]], grad_fn=<SliceBackward0>)
